In [1]:
import numpy as np
import pandas as pd
import math
import json
from tqdm.notebook import tqdm
import warnings

# --- Importiamo TUTTI i modelli per il Super Stack ---
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV # <-- NEW IMPORT
warnings.filterwarnings('ignore')
from catboost import CatBoostClassifier

# --- Definiamo i percorsi ai dati ---
train_file_path = '../input/fds-pokemon-battles-prediction-2025/train.jsonl'
test_file_path = '../input/fds-pokemon-battles-prediction-2025/test.jsonl'

I loaded the data beforehand to process a little bit.

In [2]:

train_data_raw = [] # Rinominiamo per chiarezza
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, 'r') as f:
        for line in f:
            train_data_raw.append(json.loads(line))
    print(f"Successfully loaded {len(train_data_raw)} train battles.")

    test_data_raw = []
    with open(test_file_path, 'r') as f:
        for line in f:
            test_data_raw.append(json.loads(line))
    print(f"Successfully loaded {len(test_data_raw)} test battles.")
    
    # --- NUOVO V33: Rimuoviamo la riga 4877 ---
    # "Linea 4877" corrisponde all'indice 4876 (se contiamo da 1)
    # o 4877 (se contiamo da 0). Rimuoviamo l'indice 4877.
    if len(train_data_raw) > 4877:
        del train_data_raw[4877]
        print(f"RIMOSSA riga 4877 (indice 4877). Il training set ora ha {len(train_data_raw)} campioni.")
    else:
        print("Warning: Impossibile rimuovere la riga 4877, il dataset è troppo piccolo.")
        
    # Rinominiamo i dati puliti
    train_data = train_data_raw
    test_data = test_data_raw

except FileNotFoundError:
    print(f"ERROR: Could not find data files at '{train_file_path}' or '{test_file_path}'.")

Loading data from '../input/fds-pokemon-battles-prediction-2025/train.jsonl'...
Successfully loaded 10000 train battles.
Successfully loaded 5000 test battles.
RIMOSSA riga 4877 (indice 4877). Il training set ora ha 9999 campioni.


In [3]:
# Cella 4 (SOSTITUITA): Creazione pk_vals + TE (V18 + V21)
from collections import defaultdict

def build_db_and_te_maps(data_list, n_splits=10):
    """
    Combina la logica originale della Cella 4 (pk_vals + losers)
    con la logica OOF TE V21-Lite (pokemon_win_rate_map).
    """
    pokemon_db = {}
    play_win_stats = {} # Per 'losers'
    pokemon_battles = [] # Per 'pokemon_win_rate_map'
    global_mean = np.mean([int(b.get("player_won", 0)) for b in data_list])

    for i, battle in enumerate(tqdm(data_list, desc="Building DBs (pk_vals + TE V18/V21) ")):
        battle_id = battle.get("battle_id", i)
        
        # --- 1. Logica pk_vals (Tua originale) ---
        all_pks = battle.get('p1_team_details', [])
        if battle.get("p2_lead_details"):
             all_pks.append(battle.get("p2_lead_details"))

        for p in all_pks:
            name = p.get("name")
            if name and name.lower() not in pokemon_db:
                pokemon_db[name.lower()] = {
                    "base_hp": p.get("base_hp", 0),
                    "base_atk": p.get("base_atk", 0),
                    "base_def": p.get("base_def", 0),
                    "base_spa": p.get("base_spa", 0),
                    "base_spd": p.get("base_spd", 0),
                    "base_spe": p.get("base_spe", 0),
                    "types": [t.lower() for t in p.get("types", []) if t],
                }
        
        # --- 2. Logica 'losers' (Tua originale) ---
        h = set([t.get("p2_pokemon_state", {}).get("name","unknown") for t in battle.get("battle_timeline")])
        for p in battle.get('p1_team_details', []):
            name = p.get("name")
            if name and name not in h:
                if name in play_win_stats:
                    play_win_stats[name]["played"] += 1
                    if battle.get("player_won") == True:
                        play_win_stats[name]["num_of_win"] += 1
                else:
                    play_win_stats[name] = {}
                    play_win_stats[name]["played"] = 1
                    play_win_stats[name]["num_of_win"] = 1 if battle.get("player_won") else 0
        
        # --- 3. Logica TE V21-Lite ---
        p1_won = int(battle.get("player_won", 0))
        p1_team = battle.get("p1_team_details", [])
        for p in p1_team:
            pk_name = p.get("name")
            if not pk_name: continue
            pokemon_battles.append({"id": battle_id, "name": pk_name, "won": p1_won})
        p2_lead = battle.get("p2_lead_details")
        if p2_lead and p2_lead.get("name"):
            pk_name = p2_lead.get("name")
            pokemon_battles.append({"id": battle_id, "name": pk_name, "won": 1 - p1_won})

    print(f"Database 'pk_vals' creato con {len(pokemon_db)} Pokémon.")
    
    # --- Calcolo finale 'losers' ---
    for key in play_win_stats:
        if play_win_stats[key]["played"] > 0:
             play_win_stats[key]["ratio"] = play_win_stats[key]["num_of_win"] / play_win_stats[key]["played"]
        else:
             play_win_stats[key]["ratio"] = 0.5
    loser_pokemons = [name for name, stats in play_win_stats.items() if stats["ratio"] < 0.4]
    print(f"Lista 'losers' creata con {len(loser_pokemons)} Pokémon.")

    # --- Calcolo finale 'pokemon_win_rate_map' (OOF) ---
    df_pk = pd.DataFrame(pokemon_battles)
    print(f"Calcolo OOF TE per {len(df_pk['name'].unique())} Pokémon...")
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_map = pd.Series(index=df_pk.index, dtype=float)
    k = 10 # Smoothing
    
    for train_idx, val_idx in skf.split(df_pk, df_pk['name']):
        train_fold = df_pk.iloc[train_idx]
        map_fold = train_fold.groupby('name')['won'].agg(['mean', 'count'])
        map_fold['smoothed_mean'] = (map_fold['mean'] * map_fold['count'] + global_mean * k) / (map_fold['count'] + k)
        oof_map.iloc[val_idx] = df_pk.iloc[val_idx]['name'].map(map_fold['smoothed_mean'])
    
    df_pk['oof_win_rate'] = oof_map.fillna(global_mean)
    final_pokemon_map = df_pk.groupby('name')['oof_win_rate'].mean()
    final_pokemon_map['_GLOBAL_MEAN_'] = global_mean
    
    print("Mappa TE V21-Lite (Lead) creata con successo.")
    return pokemon_db, loser_pokemons, final_pokemon_map

# Eseguiamo la funzione
if 'train_data' in locals():
    # Ora abbiamo TUTTI e 3 gli output
    pk_vals, losers, pokemon_win_rate_map = build_db_and_te_maps(train_data)
else:
    print("Dati 'train_data' non trovati.")

Building DBs (pk_vals + TE V18/V21) :   0%|          | 0/9999 [00:00<?, ?it/s]

Database 'pk_vals' creato con 20 Pokémon.
Lista 'losers' creata con 4 Pokémon.
Calcolo OOF TE per 20 Pokémon...
Mappa TE V21-Lite (Lead) creata con successo.


In [4]:
#import matplotlib.pyplot as plt
#print([(x - a[i - 1])*100 for i, x in enumerate(a)][1:])
#plt.plot(a,"o")

In [5]:
# Cella 6 (SOSTITUITA): Feature Engineering V35 (V33 + PDF Crit/Wrap/Tier)
from collections import defaultdict
import numpy as np
import pandas as pd
import math
from tqdm.notebook import tqdm

# --- HELPER 1: PARSING DELLA BATTLE TIMELINE (V28) ---
def parse_battle_timeline(timeline):
    p1_last_hp = 1.0; p1_last_status = 'ok'; p1_last_boost_sum = 0
    p2_last_hp = 1.0; p2_last_status = 'ok'; p2_last_boost_sum = 0
    if not isinstance(timeline, list) or len(timeline) == 0:
        return 0.0, 0, 0 
    found_p1 = False; found_p2 = False
    for turn_data in reversed(timeline):
        if not isinstance(turn_data, dict): continue
        if not found_p1 and 'p1_pokemon_state' in turn_data and turn_data['p1_pokemon_state']:
            state = turn_data['p1_pokemon_state']
            p1_last_hp = state.get('hp_pct', p1_last_hp)
            p1_last_status = (state.get('status', 'ok') or 'ok').lower()
            if state.get('boosts') and isinstance(state['boosts'], dict):
                p1_last_boost_sum = sum(state['boosts'].values())
            found_p1 = True
        if not found_p2 and 'p2_pokemon_state' in turn_data and turn_data['p2_pokemon_state']:
            state = turn_data['p2_pokemon_state']
            p2_last_hp = state.get('hp_pct', p2_last_hp)
            p2_last_status = (state.get('status', 'ok') or 'ok').lower()
            if state.get('boosts') and isinstance(state['boosts'], dict):
                p2_last_boost_sum = sum(state['boosts'].values())
            found_p2 = True
        if found_p1 and found_p2: break
    status_map = {'slp': 3, 'frz': 3, 'par': 2, 'psn': 1, 'tox': 1, 'brn': 1, 'ok': 0}
    p1_status_score = status_map.get(p1_last_status, 0)
    p2_status_score = status_map.get(p2_last_status, 0)
    p1_status_adv = p1_status_score - p2_status_score
    return (p1_last_hp - p2_last_hp, p1_last_boost_sum - p2_last_boost_sum, p1_status_adv)

# --- FUNZIONE DI FEATURE ENGINEERING PRINCIPALE (V35) ---
def create_golden_features_V35(data: list[dict], pk_vals, losers_list, te_map, is_test=False):
    """
    Questa è la V35:
    - V33 (0.84588)
    - + V35 (Crit Rate diff)
    - + V35 (Wrap move diff)
    - + V35 (S-Tier List Corretta, +Exeggutor)
    """
    pokemon_win_rate_map = te_map
    global_pk_mean = pokemon_win_rate_map.get('_GLOBAL_MEAN_', 0.5)
    
    # --- [MODIFICATO V35] Lista Tier (ora include Exeggutor) ---
    S_TIER_POKEMON = {'snorlax', 'chansey', 'tauros', 'exeggutor'}
    
    HYPER_BEAM_MOVES = {'hyper-beam'}
    EXPLOSION_MOVES = {'explosion', 'self-destruct'}
    OP_BOOST_MOVES = {'amnesia', 'swords-dance'}
    OP_STATUS_MOVES = {'sleep-powder', 'thunder-wave', 'sing', 'spore', 'lovelykiss', 'glare'}
    
    # --- [NUOVO V35] Mosse "Partial Trapping" ---
    PARTIAL_TRAP_MOVES = {'wrap', 'bind', 'fire-spin', 'clamp'}

    # ---- Gen I type chart & Moves (dalla tua V18) ----
    TYPE_CHART = { "normal": {"rock": 0.5, "ghost": 0.0}, "fire": {"fire": 0.5, "water": 0.5, "grass": 2, "ice": 2, "bug": 2, "rock": 0.5, "dragon": 0.5}, "water": {"fire": 2, "water": 0.5, "grass": 0.5, "ground": 2, "rock": 2, "dragon": 0.5}, "electric": {"water": 2, "electric": 0.5, "grass": 0.5, "ground": 0, "flying": 2, "dragon": 0.5}, "grass": {"fire": 0.5, "water": 2, "grass": 0.5, "poison": 0.5, "ground": 2, "flying": 0.5, "bug": 0.5, "rock": 2, "dragon": 0.5}, "ice": {"water": 0.5, "grass": 2, "ice": 0.5, "ground": 2, "flying": 2, "dragon": 2}, "fighting": {"normal": 2, "ice": 2, "poison": 0.5, "flying": 0.5, "psychic": 0.5, "bug": 0.5, "rock": 2, "ghost": 0}, "poison": {"grass": 2, "poison": 0.5, "ground": 0.5, "bug": 2, "rock": 0.5, "ghost": 0.5}, "ground": {"fire": 2, "electric": 2, "grass": 0.5, "poison": 2, "flying": 0, "bug": 0.5, "rock": 2}, "flying": {"electric": 0.5, "grass": 2, "fighting": 2, "bug": 2, "rock": 0.5}, "psychic": {"fighting": 2, "poison": 2, "psychic": 0.5, "ghost": 0.0}, "bug": {"fire": 0.5, "grass": 2, "fighting": 0.5, "poison": 2, "flying": 0.5, "psychic": 0.5, "ghost": 0.5}, "rock": {"fire": 2, "ice": 2, "fighting": 0.5, "ground": 0.5, "flying": 2, "bug": 2}, "ghost": {"normal": 0, "psychic": 0.0, "ghost": 2}, "dragon": {"dragon": 2}, "notype": {} }
    TYPE_CHART.setdefault("bug", {}).update({"poison": 2.0}); TYPE_CHART.setdefault("poison", {}).update({"bug": 2.0})
    KNOWN_TYPES = set(TYPE_CHART.keys())
    STATUS_MOVES = {"sleeppowder", "sing", "stunspore", "thunderwave", "toxic", "glare", "poisonpowder", "spore", "supersonic"}
    BOOST_MOVES = {"amnesia", "swordsdance", "agility", "barrier", "doubleteam", "growth", "harden", "meditate", "minimize"}
    FIXED_DAMAGE_MOVES = {"seismictoss", "nightshade", "sonicboom", "psywave"}
    OHKO_MOVES = {"fissure", "horndrill", "guillotine"}
    DIRECT_HEALING_MOVES = {"recover", "softboiled"}
    REST_MOVE = {"rest"}
    DRAIN_MOVES = {"megadrain"}
    T1_STATUS_MOVES = {"sleeppowder", "spore", "thunderwave", "toxic"}
    T1_RECOVERY_MOVES = {"recover", "softboiled", "rest"}
    
    def _clean_types(ts): return [(t or "").lower() if (t or "").lower() in KNOWN_TYPES else "notype" for t in (ts or [])]
    def _type_mult(attacking, defending_types):
        attacking = (attacking or "").lower(); out = 1.0
        if not defending_types: return 1.0
        table = TYPE_CHART.get(attacking, {})
        for d in defending_types: out *= table.get((d or "").lower(), 1.0)
        return out
    def _stab(move_type, attacker_types):
        m = (move_type or "").lower()
        return 1.5 if m and any(t.lower() == m for t in (attacker_types or [])) else 1.0
    def _exp_damage(move, atk_types, def_types, cap=200.0):
        if not move: return 0.0
        name = (move.get("name") or "").lower(); base = float(move.get("base_power", 0) or 0.0)
        acc = float(move.get("accuracy", 1.0) or 1.0); mtype = (move.get("type") or "").lower()
        if name in FIXED_DAMAGE_MOVES: return min(100.0 * acc, cap)
        if name in OHKO_MOVES: return min(250.0 * acc, cap)
        val = base * acc * _type_mult(mtype, def_types) * _stab(mtype, atk_types)
        return min(val, cap)

    rows = []
    pbar_desc = "Extracting V35 (Crit/Wrap) features"
    if is_test: pbar_desc = "Extracting V35 (Test) features"

    for b in tqdm(data, desc=pbar_desc):
        
        p2_team_seen = {}; p1_played_team = {}
        if b.get("battle_timeline"):
            for turn in b.get("battle_timeline"):
                d_name = turn.get("p2_pokemon_state", {}).get("name", "unknown")
                if d_name not in p2_team_seen and d_name in pk_vals: 
                    p2_team_seen[d_name] = pk_vals[d_name]
                d_name2 = turn.get("p1_pokemon_state", {}).get("name", "unknown")
                if d_name2 not in p1_played_team and d_name2 in pk_vals:
                    p1_played_team[d_name2] = pk_vals[d_name2]
        
        p1_num_of_losers = sum(1 for name in p1_played_team if name in losers_list)
        p2_num_of_losers = sum(1 for name in p2_team_seen if name in losers_list)
        loser_diff = p1_num_of_losers - p2_num_of_losers

        r = {}
        p1_team = b.get("p1_team_details") or []
        p1_lead = (b.get("p1_lead_details") or {})
        p2_lead = (b.get("p2_lead_details") or {})
        tl = b.get("battle_timeline") or []
        n_turns = len(tl) or 1
        
        # --- Feature Statiche V19 ---
        if p1_team:
            r["p1_mean_hp"] = float(np.mean([p.get("base_hp", 0) for p in p1_team]))
            r["p1_mean_atk"] = float(np.mean([p.get("base_atk", 0) for p in p1_team]))
            r["p1_mean_def"] = float(np.mean([p.get("base_def", 0) for p in p1_team]))
            r["p1_mean_spa"] = float(np.mean([p.get("base_spa", 0) for p in p1_team]))
            r["p1_mean_spd"] = float(np.mean([p.get("base_spd", 0) for p in p1_team]))
            r["p1_mean_spe"] = float(np.mean([p.get("base_spe", 0) for p in p1_team]))
        else:
            r["p1_mean_hp"]=0.0; r["p1_mean_atk"]=0.0; r["p1_mean_def"]=0.0; r["p1_mean_spa"]=0.0; r["p1_mean_spd"]=0.0; r["p1_mean_spe"]=0.0
        
        if p2_team_seen:
            r["p2_mean_hp"] = float(np.mean([p2_team_seen[p].get("base_hp", 0) for p in p2_team_seen]))
            r["p2_mean_atk"] = float(np.mean([p2_team_seen[p].get("base_atk", 0) for p in p2_team_seen]))
            r["p2_mean_def"] = float(np.mean([p2_team_seen[p].get("base_def", 0) for p in p2_team_seen]))
            r["p2_mean_spa"] = float(np.mean([p2_team_seen[p].get("base_spa", 0) for p in p2_team_seen]))
            r["p2_mean_spd"] = float(np.mean([p2_team_seen[p].get("base_spd", 0) for p in p2_team_seen]))
            r["p2_mean_spe"] = float(np.mean([p2_team_seen[p].get("base_spe", 0) for p in p2_team_seen]))
        else:
            r["p2_mean_hp"]=0.0; r["p2_mean_atk"]=0.0; r["p2_mean_def"]=0.0; r["p2_mean_spa"]=0.0; r["p2_mean_spd"]=0.0; r["p2_mean_spe"]=0.0

        p1_lead_stats = defaultdict(int, pk_vals.get(p1_lead.get('name', '').lower(), {}))
        p2_lead_stats = defaultdict(int, pk_vals.get(p2_lead.get('name', '').lower(), {}))

        r["lead_hp_diff"] = p1_lead_stats['base_hp'] - p2_lead_stats['base_hp']
        r["lead_atk_diff"] = p1_lead_stats['base_atk'] - p2_lead_stats['base_atk']
        r["lead_def_diff"] = p1_lead_stats['base_def'] - p2_lead_stats['base_def']
        r["lead_spa_diff"] = p1_lead_stats['base_spa'] - p2_lead_stats['base_spa']
        r["lead_spd_diff"] = p1_lead_stats['base_spd'] - p2_lead_stats['base_spd']
        r["lead_speed_diff"] = p1_lead_stats['base_spe'] - p2_lead_stats['base_spe']
        
        r["mean_hp_diff"] = r["p1_mean_hp"] - r["p2_mean_hp"]
        r["mean_atk_diff"] = r["p1_mean_atk"] - r["p2_mean_atk"]
        r["mean_def_diff"] = r["p1_mean_def"] - r["p2_mean_def"]
        r["mean_spa_diff"] = r["p1_mean_spa"] - r["p2_mean_spa"]
        r["mean_spd_diff"] = r["p1_mean_spd"] - r["p2_mean_spd"]
        r["mean_speed_diff"] = r["p1_mean_spe"] - r["p2_mean_spe"]
        
        r["loser_diff"] = loser_diff
        
        # --- [NUOVO V35] Crit Rate (Static) ---
        p1_crit_rates = [p.get("base_spe", 0) / 512.0 for p in p1_team]
        p2_crit_rates = [p2_team_seen[p].get("base_spe", 0) / 512.0 for p in p2_team_seen]
        r["p1_mean_crit_rate"] = np.mean(p1_crit_rates) if p1_crit_rates else 0
        r["p2_mean_crit_rate"] = np.mean(p2_crit_rates) if p2_crit_rates else 0
        r["crit_rate_diff"] = r["p1_mean_crit_rate"] - r["p2_mean_crit_rate"]
        # --- Fine V35 ---

        # --- Logica Timeline V18 + V28 + V32 + V33 + V35 ---
        mid_hp_diff = 0.0; final_boost_advantage = 0.0; p1_exp, p2_exp = 0.0, 0.0; p1_sw, p2_sw = 0, 0
        p1_remaining_pokemons = {} 
        p1_team_status = {} 
        p2_team_status = {}
        p1_stat, p2_stat = 0, 0; p1_boost, p2_boost = 0, 0; p1_dmg, p2_dmg = 0, 0; p1_lost, p2_lost = 0.0, 0.0; p1_slp = p2_slp = 0; p1_frz = p2_frz = 0; p1_psn = p2_psn = 0; p1_par = p2_par = 0; p1_brn = p2_brn = 0; p1_dheal = p2_dheal = 0.0; p1_drain = p2_drain = 0.0
        p1_t1_status, p2_t1_status = 0, 0; p1_t1_recover, p2_t1_recover = 0, 0
        
        mid_boost_diff = 0.0; mid_faint_diff = 0.0; mid_status_adv = 0.0
        status_map = {'slp': 3, 'frz': 3, 'par': 2, 'psn': 1, 'tox': 1, 'brn': 1, 'ok': 0}
        
        p1_hyper_beam_ko = 0; p2_hyper_beam_ko = 0
        p1_explosion_trade = 0; p2_explosion_trade = 0
        
        p1_op_boost_count = 0; p2_op_boost_count = 0
        p1_op_status_count = 0; p2_op_status_count = 0
        
        # --- [NUOVO V35] Contatori Wrap ---
        p1_wrap_count = 0
        p2_wrap_count = 0

        if len(tl) > 0:
            mid_idx = len(tl) // 2
            if tl[mid_idx].get("p1_pokemon_state") and tl[mid_idx].get("p2_pokemon_state"):
                mid_hp_diff = (tl[mid_idx].get("p1_pokemon_state", {}).get("hp_pct", 0.0) - tl[mid_idx].get("p2_pokemon_state", {}).get("hp_pct", 0.0))
            
            ps1_last_valid = {}; ps2_last_valid = {}
            
            for i, turn in enumerate(tl):
                if i == mid_idx:
                    p1_mid_boost = sum((turn.get("p1_pokemon_state", {}).get("boosts", {}) or {}).values())
                    p2_mid_boost = sum((turn.get("p2_pokemon_state", {}).get("boosts", {}) or {}).values())
                    mid_boost_diff = p1_mid_boost - p2_mid_boost
                    p1_mid_faint = sum(1 for p in p1_team_status.values() if p['fainted'])
                    p2_mid_faint = sum(1 for p in p2_team_status.values() if p['fainted'])
                    mid_faint_diff = p1_mid_faint - p2_mid_faint
                    s1_mid = (turn.get("p1_pokemon_state", {}).get("status", "ok") or "ok").lower()
                    s2_mid = (turn.get("p2_pokemon_state", {}).get("status", "ok") or "ok").lower()
                    mid_status_adv = status_map.get(s1_mid, 0) - status_map.get(s2_mid, 0)

                ps1 = turn.get("p1_pokemon_state") or {}; ps2 = turn.get("p2_pokemon_state") or {}; s1 = ps1.get("status", "nostatus"); s2 = ps2.get("status", "nostatus")
                if ps1 and ps1.get("name"):
                    ps1_last_valid = ps1 
                    hp = ps1.get("hp_pct", 0.0)
                    p1_team_status[ps1.get("name")] = {'hp': hp, 'fainted': (hp <= 0.0)}
                    if 'hp_pct' not in p1_remaining_pokemons.get(ps1.get("name"), {}):
                         p1_remaining_pokemons[ps1.get("name")] = {}
                    p1_remaining_pokemons[ps1.get("name")]["hp_pct"] = hp
                if ps2 and ps2.get("name"):
                    ps2_last_valid = ps2
                    hp = ps2.get("hp_pct", 0.0)
                    p2_team_status[ps2.get("name")] = {'hp': hp, 'fainted': (hp <= 0.0)}
                
                if s1 == "slp": p1_slp += 1
                elif s1 == "frz": p1_frz += 1
                elif s1 in ("psn", "tox"): p1_psn += 1
                elif s1 == "par": p1_par += 1
                elif s1 == "brn": p1_brn += 1
                if s2 == "slp": p2_slp += 1
                elif s2 == "frz": p2_frz += 1
                elif s2 in ("psn", "tox"): p2_psn += 1
                elif s2 == "par": p2_par += 1
                elif s2 == "brn": p2_brn += 1
                if i > 0:
                    prev1 = tl[i - 1].get("p1_pokemon_state") or {}; prev2 = tl[i - 1].get("p2_pokemon_state") or {}
                    if ps1.get("name") == prev1.get("name"): h1, h1p = ps1.get("hp_pct", 1.0) or 0.0, prev1.get("hp_pct", 1.0) or 0.0; p1_lost += (h1p - h1) if h1 < h1p else 0
                    if ps2.get("name") == prev2.get("name"): h2, h2p = ps2.get("hp_pct", 1.0) or 0.0, prev2.get("hp_pct", 1.0) or 0.0; p2_lost += (h2p - h2) if h2 < h2p else 0
                if turn.get("p1_move_details") is None: p1_sw += 1
                if turn.get("p2_move_details") is None: p2_sw += 1
                m1 = turn.get("p1_move_details"); m2 = turn.get("p2_move_details")
                
                n1 = (m1.get("name") or "").lower() if m1 else ""
                n2 = (m2.get("name") or "").lower() if m2 else ""
                
                if ps2.get('hp_pct') == 0.0 or ps2.get('status') == 'fnt':
                    if n1 in HYPER_BEAM_MOVES: p1_hyper_beam_ko += 1
                    if n1 in EXPLOSION_MOVES: p1_explosion_trade += 1
                if ps1.get('hp_pct') == 0.0 or ps1.get('status') == 'fnt':
                    if n2 in HYPER_BEAM_MOVES: p2_hyper_beam_ko += 1
                    if n2 in EXPLOSION_MOVES: p2_explosion_trade += 1
                
                if m1 and ps2:
                    atk = _clean_types(ps1.get("types", [])); dft = _clean_types(ps2.get("types", [])); p1_exp += _exp_damage(m1, atk, dft)
                    if n1 in OP_BOOST_MOVES: p1_op_boost_count += 1
                    if n1 in OP_STATUS_MOVES: p1_op_status_count += 1
                    if n1 in PARTIAL_TRAP_MOVES: p1_wrap_count += 1 # V35
                    if n1 in DIRECT_HEALING_MOVES: p1_dheal += 0.5
                    elif n1 in DRAIN_MOVES: p1_drain += 0.1
                    if n1 in STATUS_MOVES: p1_stat += 1 
                    elif n1 in BOOST_MOVES: p1_boost += 1 
                    elif (m1.get("base_power", 0) or 0) > 0: p1_dmg += 1
                    if n1 in T1_STATUS_MOVES: p1_t1_status += 1
                    if n1 in T1_RECOVERY_MOVES: p1_t1_recover += 1
                if m2 and ps1:
                    atk = _clean_types(ps2.get("types", [])); dft = _clean_types(ps1.get("types", [])); p2_exp += _exp_damage(m2, atk, dft)
                    if n2 in OP_BOOST_MOVES: p2_op_boost_count += 1
                    if n2 in OP_STATUS_MOVES: p2_op_status_count += 1
                    if n2 in PARTIAL_TRAP_MOVES: p2_wrap_count += 1 # V35
                    if n2 in DIRECT_HEALING_MOVES: p2_dheal += 0.5
                    elif n2 in DRAIN_MOVES: p2_drain += 0.1
                    if n2 in STATUS_MOVES: p2_stat += 1 
                    elif n2 in BOOST_MOVES: p2_boost += 1 
                    elif (m2.get("base_power", 0) or 0) > 0: p2_dmg += 1
                    if n2 in T1_STATUS_MOVES: p2_t1_status += 1
                    if n2 in T1_RECOVERY_MOVES: p2_t1_recover += 1
            
            final_boost_advantage = sum((ps1_last_valid.get("boosts", {}) or {}).values()) - sum((ps2_last_valid.get("boosts", {}) or {}).values())

        p1_all_names = [p.get("name", "unknown") for p in p1_team]
        for name in p1_all_names: 
            if name not in p1_remaining_pokemons.keys():
                p1_remaining_pokemons[name] = {}
                p1_remaining_pokemons[name]["hp_pct"] = 1.0
        
        p1_remaining_pk = {name: val for name, val in p1_remaining_pokemons.items() if val.get("hp_pct", 0) > 0}
        p2_remaining_pk = {name: val for name, val in p2_team_status.items() if not val.get("fainted", True)}
        
        p1_remaining_speed_sum = 0
        for k,v in p1_remaining_pk.items():
            if k in pk_vals:
                p1_remaining_pk[k]["rem_hp"] = p1_remaining_pk[k]["hp_pct"] * pk_vals[k].get("base_hp", 0)
                p1_remaining_pk[k]["base_spe"] = pk_vals[k].get("base_spe", 0)
                p1_remaining_speed_sum += p1_remaining_pk[k]["base_spe"]
        
        r["remaining_total_hp"] = np.sum([p1_remaining_pk[p].get("rem_hp", 0) for p in p1_remaining_pk.keys()])
        r["p1_remaining_speed_sum"] = p1_remaining_speed_sum
        
        p2_remaining_speed_sum = 0
        for k in p2_remaining_pk.keys():
             if k in pk_vals:
                p2_remaining_speed_sum += pk_vals[k].get("base_spe", 0)
        r["p2_remaining_speed_sum"] = p2_remaining_speed_sum
        r["remaining_speed_diff"] = r["p1_remaining_speed_sum"] - r["p2_remaining_speed_sum"]
        
        p1_fnt_count = sum(1 for p in p1_team_status.values() if p['fainted'])
        p2_fnt_count = sum(1 for p in p2_team_status.values() if p['fainted'])
        r["fnt_count_diff"] = p1_fnt_count - p2_fnt_count
        
        # --- [MODIFICATO V35] Usa la S_TIER_POKEMON list (con Exeggutor) ---
        p1_fnt_hitters = sum(1 for name, state in p1_team_status.items() if state['fainted'] and name in S_TIER_POKEMON)
        p2_fnt_hitters = sum(1 for name, state in p2_team_status.items() if state['fainted'] and name in S_TIER_POKEMON)
        r["heavy_hitter_faint_diff"] = p2_fnt_hitters - p1_fnt_hitters 

        p1_total_hp = sum(p['hp'] for p in p1_team_status.values() if not p['fainted'])
        p2_total_hp = sum(p['hp'] for p in p2_team_status.values() if not p['fainted'])
        p1_unseen_count = 6 - len(p1_team_status)
        p2_unseen_count = 6 - len(p2_team_status)
        r["team_hp_total_p1"] = p1_total_hp + p1_unseen_count
        r["team_hp_total_p2"] = p2_total_hp + p2_unseen_count
        r["team_hp_total_diff"] = r["team_hp_total_p1"] - r["team_hp_total_p2"]
        
        r["mid_hp_diff"] = float(mid_hp_diff)
        r["final_boost_advantage"] = float(final_boost_advantage) 
        r["expected_power_diff"] = float(max(min(p1_exp - p2_exp, 2000.0), -2000.0))
        r["switch_diff"] = float(p1_sw - p2_sw) / n_turns
        r["pokemon_seen_diff"] = float(len(p1_team_status)) - float(len(p2_team_status)) / n_turns
        r["status_moves_diff"] = float(p1_stat - p2_stat) / n_turns 
        r["boost_moves_diff"] = float(p1_boost - p2_boost) / n_turns 
        r["damage_moves_diff"] = float(p1_dmg - p2_dmg) / n_turns
        r["total_hp_lost_diff"] = float(p2_lost - p1_lost)
        r["slp_frz_turns_adv"] = float((p2_slp + p2_frz) - (p1_slp + p1_frz)) / n_turns
        r["psn_tox_turns_adv"] = float(p2_psn - p1_psn) / n_turns
        r["lost_speed_turns_adv"] = float(p2_par - p1_par) / n_turns
        r["reduced_atk_turns_adv"] = float(p2_brn - p1_brn) / n_turns
        r["healing_adv"] = float((p1_dheal + p1_drain) - (p2_dheal + p2_drain)) / n_turns
        r["t1_status_adv"] = float(p1_t1_status - p2_t1_status) / n_turns
        r["t1_recover_adv"] = float(p1_t1_recover - p2_t1_recover) / n_turns
        
        p1_lead_wr = pokemon_win_rate_map.get(p1_lead.get('name', '').lower(), global_pk_mean)
        p2_lead_wr = pokemon_win_rate_map.get(p2_lead.get('name', '').lower(), global_pk_mean)
        r['lead_win_rate_diff'] = p1_lead_wr - p2_lead_wr
        (r['last_hp_pct_diff'], r['last_boost_sum_diff'], r['p1_status_adv']) = parse_battle_timeline(tl)
        p1_played_wr = [pokemon_win_rate_map.get(name.lower(), global_pk_mean) for name in p1_played_team]
        p2_seen_wr = [pokemon_win_rate_map.get(name.lower(), global_pk_mean) for name in p2_team_seen]
        r['p1_played_team_mean_wr'] = np.mean(p1_played_wr) if p1_played_wr else global_pk_mean
        r['p2_seen_team_mean_wr'] = np.mean(p2_seen_wr) if p2_seen_wr else global_pk_mean
        r['played_team_wr_diff'] = r['p1_played_team_mean_wr'] - r['p2_seen_team_mean_wr']
        
        r['mid_boost_diff'] = mid_boost_diff
        r['mid_faint_diff'] = mid_faint_diff
        r['mid_status_adv'] = mid_status_adv
        
        r['hyper_beam_ko_diff'] = (p1_hyper_beam_ko - p2_hyper_beam_ko) / n_turns
        r['explosion_trade_diff'] = (p1_explosion_trade - p2_explosion_trade) / n_turns
        
        r['op_boost_diff'] = (p1_op_boost_count - p2_op_boost_count) / n_turns
        r['op_status_diff'] = (p1_op_status_count - p2_op_status_count) / n_turns
        
        # --- 6. AGGIUNTA FEATURE V35 (NUOVE) ---
        r['wrap_move_diff'] = (p1_wrap_count - p2_wrap_count) / n_turns
        
        r["battle_id"] = b.get("battle_id")
        if not is_test:
            r["player_won"] = int(b["player_won"])
        
        rows.append(r)
    
    df = pd.DataFrame(rows).fillna(0)
    for c in df.select_dtypes(include=["float"]).columns: df[c] = df[c].astype("float32")
    for c in df.select_dtypes(include=["int"]).columns:
        if c != "player_won": df[c] = df[c].astype("int32")
    return df

In [6]:
# Cella 7 (SOSTITUITA): Processing con V33 (Correzione KeyError)

if 'train_data' in locals() and 'pokemon_win_rate_map' in locals() and 'pk_vals' in locals() and 'losers' in locals():
    print("\nProcessing training data (V33)...")
    train_df = create_golden_features_V35(
        train_data, pk_vals, losers, pokemon_win_rate_map, is_test=False
    )
    
    print("\nProcessing test data (V33)...")
    test_df = create_golden_features_V35(
        test_data, pk_vals, losers, pokemon_win_rate_map, is_test=True
    )
    
    # Drop the target from test_df
    if 'player_won' in test_df.columns:
        test_df = test_df.drop(columns=['player_won'])
    
    print(f"\nCreated {len(train_df.columns) - 2} features.")
    
    # --- Allineamento Colonne (LOGICA CORRETTA) ---
    # Definiamo le feature (X) e il target (y)
    target = 'player_won'
    features_v33 = [col for col in train_df.columns if col not in ['battle_id', target]]
    
    # Creiamo i DataFrame di training e test (X)
    X_train_full = train_df[features_v33]
    y_train_full = train_df[target]
    
    # Allineiamo il test set
    for col in features_v33:
        if col not in test_df:
            print(f"Warning: Aggiunta colonna mancante '{col}' al test set.")
            test_df[col] = 0
            
    # X_test_full contiene solo le feature per il modello
    X_test_full = test_df[features_v33]
    
    # 'test_df' (l'originale) rimane intatto e CONTIENE 'battle_id'
    # 'train_df' (l'originale) rimane intatto e CONTIENE 'battle_id' e 'player_won'
    
    print("Allineamento completato. 'test_df' contiene 'battle_id'.")
    
else:
    print("\nSkipping feature processing. Esegui Cella 3 e 4 prima.")


Processing training data (V33)...


Extracting V35 (Crit/Wrap) features:   0%|          | 0/9999 [00:00<?, ?it/s]


Processing test data (V33)...


Extracting V35 (Test) features:   0%|          | 0/5000 [00:00<?, ?it/s]


Created 68 features.
Allineamento completato. 'test_df' contiene 'battle_id'.


In [7]:
if 'train_df' in locals():
    # --- Define our "Level 1" UN-TUNED Base Models ---
    # We use the un-tuned models because we proved they stack better
    model_xgb = Pipeline([
        ('model', xgb.XGBClassifier(
            colsample_bytree=0.8, learning_rate=0.1, max_depth=3,
            n_estimators=250, subsample=0.7, random_state=42,
            eval_metric='logloss', use_label_encoder=False, n_jobs=-1
        ))
    ])
    model_gb = Pipeline([
        ('model', GradientBoostingClassifier(
            n_estimators=250, random_state=42, max_depth=3, learning_rate=0.1
        ))
    ])
    model_lr = Pipeline([
        ('scaler', StandardScaler()), 
        ('model', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
    ])

    model_cb = Pipeline([
            ('model', CatBoostClassifier(
            iterations = 1000, random_state=42, max_depth=4, learning_rate=0.1, verbose = 0
        ))
    ])
    base_models = { 'xgb': model_xgb, 'gb': model_gb, 'lr': model_lr , "cb":model_cb}

    print("\n--- Starting Stacking (on V22 ---")
    
    # --- Define features and target ---
    target = 'player_won'
    features_v5 = [col for col in train_df.columns if col not in ['battle_id', target]] 
    X_train_v5 = train_df[features_v5]
    y_train = train_df[target]
    X_test_v5 = test_df[features_v5]
    
    # Align columns just in case
    X_test_v5 = X_test_v5[X_train_v5.columns]

    meta_features_train = pd.DataFrame()
    meta_features_test = pd.DataFrame() 
    N_SPLITS_STACK = 5
    skf_stack = StratifiedKFold(n_splits=N_SPLITS_STACK, shuffle=True, random_state=42) 

    for model_name, model in base_models.items():
        print(f"Training {model_name} on V5 features...")
        
        # 1. Get OOF predictions for the training data
        oof_preds = cross_val_predict(
            model, X_train_v5, y_train, cv=skf_stack, method='predict_proba', n_jobs=-1
        )[:, 1]
        meta_features_train[f'pred_{model_name}'] = oof_preds
        
        # 2. Train on ALL V5 training data and predict on test set
        model.fit(X_train_v5, y_train)
        test_preds = model.predict_proba(X_test_v5)[:, 1]
        meta_features_test[f'pred_{model_name}'] = test_preds

    print("\n--- Level 1 OOF Predictions Generated ---")

    # --- Train the "Level 2" Meta-Model ---
    print("\n--- Training Level 2 Meta-Model ---")
    meta_model = LogisticRegression(random_state=42, n_jobs=-1)
    
    # Fit the final meta-model on all OOF predictions
    meta_model.fit(meta_features_train, y_train)


--- Starting Stacking (on V22 ---
Training xgb on V5 features...
Training gb on V5 features...
Training lr on V5 features...
Training cb on V5 features...

--- Level 1 OOF Predictions Generated ---

--- Training Level 2 Meta-Model ---


In [8]:
if 'meta_model' in locals():
    print("\n--- Finding optimal threshold for the (V35) STACKED model ---")

    # Get OOF predictions for the meta-model
    oof_probabilities_stacked = cross_val_predict(
        meta_model, 
        meta_features_train, # Train on the meta-features
        y_train, 
        cv=skf_stack, 
        method='predict_proba', 
        n_jobs=-1
    )[:, 1]
    
    print("Searching for the best accuracy threshold...")
    thresholds = np.linspace(0.3, 0.7, 81)
    accuracies = []
    for thr in thresholds:
        oof_preds_at_thr = (oof_probabilities_stacked >= thr).astype(int)
        acc = accuracy_score(y_train, oof_preds_at_thr)
        accuracies.append(acc)

    best_thr_idx = np.argmax(accuracies)
    optimal_threshold_stacked = thresholds[best_thr_idx]
    best_accuracy_stacked = accuracies[best_thr_idx]

    print(f"\nOptimal stacked threshold found: {optimal_threshold_stacked:.4f}")
    print(f"Stacked accuracy at optimal threshold: {best_accuracy_stacked:.5f}")


--- Finding optimal threshold for the (V35) STACKED model ---
Searching for the best accuracy threshold...

Optimal stacked threshold found: 0.5800
Stacked accuracy at optimal threshold: 0.84558


In [9]:
if 'meta_model' in locals():
    print(f"Models are already trained.")
    print("Predicting probabilities on the test set meta-features...")
    
    # We use the meta_model to predict on the test_set's meta-features
    test_probabilities_stacked = meta_model.predict_proba(meta_features_test)[:, 1]
    
    # Apply the OPTIMAL Stacked Threshold
    print(f"Applying optimal stacked threshold: {optimal_threshold_stacked:.4f}")
    final_predictions = (test_probabilities_stacked >= optimal_threshold_stacked).astype(int)

    # Create the Final Submission File
    print("\nCreating final submission file...")
    submission_df = pd.DataFrame({
        'battle_id': test_df['battle_id'],
        'player_won': final_predictions
    })
    submission_df.to_csv('submission.csv', index=False)
    
    print("\n'submission.csv' file created successfully!")
    display(submission_df.head())

Models are already trained.
Predicting probabilities on the test set meta-features...
Applying optimal stacked threshold: 0.5800

Creating final submission file...

'submission.csv' file created successfully!


,battle_id,player_won
0,0,0
1,1,1
2,2,1
3,3,1
4,4,1
